In [1]:
import os
import nibabel as nib
import nibabel.orientations as nio
import pydicom
import numpy as np
from tqdm import tqdm
from collections import Counter

def print_dict_counter(orientation_dict):
    values = list(orientation_dict.values())
    counts = Counter(values)

    total = len(values)

    for k, v in counts.items():
        print(f"{k}: {v/total:.4f} ({v}/{total})")

# T2 Haste

In [2]:
t2_haste_root = "/mnt/gpussd2/jrich/Desktop/ADPKD/crisp/T2_HASTE"

def return_orientation(img_path):
    img = nib.load(img_path)
    orientation = nio.aff2axcodes(img.affine)

    if orientation[2] in ['S', 'I']:
        return "axial"
    elif orientation[2] in ['A', 'P']:
        return "coronal"
    elif orientation[2] in ['L', 'R']:
        return "sagittal"
    else:
        return "unknown"

t2_haste_orientation_dict = {}

for dirpath, _, filenames in tqdm(os.walk(t2_haste_root), desc="Processing directories"):
    for f in filenames:
        if f.endswith(".nii.gz"):
            path = os.path.join(dirpath, f)
            relpath = os.path.relpath(path, t2_haste_root)
            t2_haste_orientation_dict[relpath] = return_orientation(path)

print_dict_counter(t2_haste_orientation_dict)

axial_keys = [k for k, v in t2_haste_orientation_dict.items() if v == "axial"]
print("Axial keys (first 5):", axial_keys[:5])

# t2_haste_orientation_dict

Processing directories: 785it [00:00, 1470.14it/s]

coronal: 0.9982 (545/546)
axial: 0.0018 (1/546)
Axial keys (first 5): ['291161/3_755425_1002/1002_1ST_HASTE_9MM_000000_4.nii.gz']


# CRISP Batch 1

In [3]:
crisp_batch1_root = "/run/user/1023/gvfs/smb-share:server=10.148.121.94,share=crisp/CRISP_Batch1"

def dicom_orientation(dcm_path):
    dcm = pydicom.dcmread(dcm_path, stop_before_pixels=True)

    iop = np.array(dcm.ImageOrientationPatient).astype(float)

    row = iop[:3]
    col = iop[3:]

    normal = np.cross(row, col)
    axis = np.argmax(np.abs(normal))

    if axis == 2:
        return "axial"
    elif axis == 1:
        return "coronal"
    elif axis == 0:
        return "sagittal"
    else:
        return "unknown"

crisp_batch1_orientation_dict = {}

for dirpath, _, filenames in os.walk(crisp_batch1_root):
    for f in filenames:
        if f.endswith(".dcm"):
            dcm_path = os.path.join(dirpath, f)
            relpath = os.path.relpath(dcm_path, crisp_batch1_root)

            try:
                orientation = dicom_orientation(dcm_path)
                crisp_batch1_orientation_dict[relpath] = orientation

            except Exception:
                continue

print_dict_counter(crisp_batch1_orientation_dict)

axial_keys = [k for k, v in crisp_batch1_orientation_dict.items() if v == "axial"]
print("Axial keys (first 5):", axial_keys[:5])

# crisp_batch1_orientation_dict

coronal: 0.9335 (89021/95364)
axial: 0.0206 (1964/95364)
sagittal: 0.0459 (4379/95364)
Axial keys (first 5): ['CRISP_Batch1/004555/1000/1007.dcm', 'CRISP_Batch1/004555/1000/1008.dcm', 'CRISP_Batch1/004555/1000/1009.dcm', 'CRISP_Batch1/004555/1000/1019.dcm', 'CRISP_Batch1/004555/1000/1032.dcm']
